In [29]:
import pandas as pd
import numpy as np
import re

# 업종코드 데이터 전처리
 * 01, 011, 0011 등 5자리 수가 아닌 업종코드 수 및 알파벳 코드 앞에 추가

In [34]:
df_industry = pd.read_excel("C:/Users/Woochul/Desktop/Project V/OCR Sample/data/metadata/11차 개정(2024년부터 적용).xlsx")
df_industry

,산업코드,산업내용
0,A,"농업, 임업 및 어업(01~03)"
1,01,농업
2,011,작물 재배업
3,0111,곡물 및 기타 식량작물 재배업
4,01110,곡물 및 기타 식량작물 재배업
...,...,...
2033,99,국제 및 외국기관
2034,990,국제 및 외국기관
2035,9900,국제 및 외국기관
2036,99001,주한 외국 공관


In [35]:
import pandas as pd

def make_industry_class_code(df: pd.DataFrame,
                             code_col: str = "산업코드",
                             out_col: str = "산업분류코드") -> pd.DataFrame:
    df = df.copy()

    s = df[code_col].astype(str).str.strip()

    # 알파벳 단독(예: 'A')인 행 -> 해당 알파벳을 아래로 전파
    alpha = s.where(s.str.fullmatch(r"[A-Za-z]")).str.upper().ffill()

    # 숫자코드(예: '01', '011', '0111'...)만 추출
    num = s.where(s.str.fullmatch(r"\d+"))

    # 숫자코드는 5자리로 0-padding
    num5 = num.str.zfill(5)

    # 알파벳 행은 산업분류코드 비워두고, 숫자코드 행만 생성
    df[out_col] = (alpha + num5).where(num.notna())

    return df

# 사용 예시
# df = make_industry_class_code(df)

In [36]:
df_industry_code = make_industry_class_code(df_industry)

In [37]:
df_industry_code

,산업코드,산업내용,산업분류코드
0,A,"농업, 임업 및 어업(01~03)",NaN
1,01,농업,A00001
2,011,작물 재배업,A00011
3,0111,곡물 및 기타 식량작물 재배업,A00111
4,01110,곡물 및 기타 식량작물 재배업,A01110
...,...,...,...
2033,99,국제 및 외국기관,U00099
2034,990,국제 및 외국기관,U00990
2035,9900,국제 및 외국기관,U09900
2036,99001,주한 외국 공관,U99001


In [38]:
df_industry_code = df_industry_code.dropna(subset=['산업분류코드'], how='any', axis=0).reset_index(drop=True)
df_industry_code

,산업코드,산업내용,산업분류코드
0,01,농업,A00001
1,011,작물 재배업,A00011
2,0111,곡물 및 기타 식량작물 재배업,A00111
3,01110,곡물 및 기타 식량작물 재배업,A01110
4,0112,"채소, 화훼작물 및 종묘 재배업",A00112
...,...,...,...
2012,99,국제 및 외국기관,U00099
2013,990,국제 및 외국기관,U00990
2014,9900,국제 및 외국기관,U09900
2015,99001,주한 외국 공관,U99001


In [39]:
df_industry_code.to_csv("./data/metadata/preprocess_corp_code_list(prototype).csv", encoding="cp949", index=False)

PermissionError: [Errno 13] Permission denied: './data/metadata/preprocess_corp_code_list(prototype).csv'

# 상장회사 업종코드 속성 추가

* 전자공시시스템 API서비스 고유번호 Excel File Raw Data와 상장일자 병합
* 기업공시채널 KIND 상장법인목록 Excel 파일과 업종코드 Data 병합 진행

In [31]:
path = "./data/metadata/상장법인목록.xls"

# HTML(.xls로 위장) → read_html로 로드
tables = pd.read_html(path, encoding="euc-kr")   # 또는 encoding="cp949"
kind_df = tables[0]

kind_df.head()

,회사명,시장구분,종목코드,업종,주요제품,상장일,결산월,대표자명,홈페이지,지역
0,엔비알모션,코스닥,0004V0,기타 금속 가공제품 제조업,강구 및 테이퍼롤러,2026-01-14,12월,문두성,NaN,경상남도
1,지에프아이,코스닥,493330,일반 목적용 기계 제조업,"2차전지소화안전제품, 일반소화안전제품",2026-01-14,12월,이상섭,NaN,경기도
2,삼미금속,코스닥,012210,기타 금속 가공제품 제조업,"자동차부품, 중장비부품, 선박부품, 방산부품 등",2025-12-29,12월,이남영,http://www.sammimetal.com,경상남도
3,세미파이브,코스닥,490470,"건축기술, 엔지니어링 및 관련 기술 서비스업",반도체 설계 서비스,2025-12-29,12월,조명현,http://semifive.com,경기도
4,리브스메드,코스닥,491000,의료용 기기 제조업,"다관절 다자유도 복강경 수술기구인 ArtiSential, ArtiSeal, Arti...",2025-12-24,12월,이정주,http://www.livsmed.com,경기도


In [32]:
len(kind_df)

2781

In [48]:
import pandas as pd
import re

def normalize_text(x):
    if pd.isna(x):
        return ""

    s = str(x).strip()
    s = s.replace("\n", " ")

    # 0) 한국어 표현 보정 (의미 보존용 타겟 치환)
    # 현재 확인된 예외 케이스
    s = s.replace("차체나", "차체및")

    # 필요시 확장 가능
    # s = s.replace("나", "및")  # <- 너무 공격적이면 주석 유지

    # 1) 구두점 통일: 콤마 포함 자주 나오는 기호 제거
    s = re.sub(r"[,，·ㆍ;:(){}\[\]<>\"'“”‘’]", "", s)

    # 2) 공백 정규화
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ", "")  # 완전 일치 매칭률 향상

    return s


# =========================
# 매핑용 DF (중복 유지)
# =========================
df_map = df_industry_code[["산업내용", "산업분류코드"]].copy()
df_map["key"] = df_map["산업내용"].map(normalize_text)

# =========================
# kind_df
# =========================
kind_df2 = kind_df.copy()
kind_df2["key"] = kind_df2["업종"].map(normalize_text)

# =========================
# 중복 허용 merge
# (산업내용이 여러 개인 경우 → 행 증가)
# =========================
kind_df2 = kind_df2.merge(
    df_map[["key", "산업분류코드"]],
    on="key",
    how="left"
)

# =========================
# 정리
# =========================
kind_df2.drop(columns=["key"], inplace=True)


In [49]:
kind_df2["산업분류코드"].isna().sum()

0

In [50]:
kind_df2.loc[
    kind_df2["산업분류코드"].isna(),
    ["회사명", "업종", "산업분류코드"]
]

,회사명,업종,산업분류코드


In [51]:
kind_df2

,회사명,시장구분,종목코드,업종,주요제품,상장일,결산월,대표자명,홈페이지,지역,산업분류코드
0,엔비알모션,코스닥,0004V0,기타 금속 가공제품 제조업,강구 및 테이퍼롤러,2026-01-14,12월,문두성,NaN,경상남도,C00259
1,지에프아이,코스닥,493330,일반 목적용 기계 제조업,"2차전지소화안전제품, 일반소화안전제품",2026-01-14,12월,이상섭,NaN,경기도,C00291
2,삼미금속,코스닥,012210,기타 금속 가공제품 제조업,"자동차부품, 중장비부품, 선박부품, 방산부품 등",2025-12-29,12월,이남영,http://www.sammimetal.com,경상남도,C00259
3,세미파이브,코스닥,490470,"건축기술, 엔지니어링 및 관련 기술 서비스업",반도체 설계 서비스,2025-12-29,12월,조명현,http://semifive.com,경기도,M00721
4,리브스메드,코스닥,491000,의료용 기기 제조업,"다관절 다자유도 복강경 수술기구인 ArtiSential, ArtiSeal, Arti...",2025-12-24,12월,이정주,http://www.livsmed.com,경기도,C00271
...,...,...,...,...,...,...,...,...,...,...,...
3276,경방,유가,000050,종합 소매업,"섬유류(면사,면혼방사,면직물,면혼방직물,화섬사,화섬직물) 제조,도매,수출입",1956-03-03,12월,"김준, 김담",http://www.kyungbang.co.kr,서울특별시,G00471
3277,유수홀딩스,유가,000700,회사 본부 및 경영 컨설팅 서비스업,지주사업,1956-03-03,12월,송영규,http://www.eusu-holdings.com,서울특별시,M00715
3278,한진중공업홀딩스,유가,003480,연료용 가스 제조 및 배관공급업,지주회사,1956-03-03,12월,조원국,http://www.hhic-holdings.com,경기도,D00352
3279,한진중공업홀딩스,유가,003480,연료용 가스 제조 및 배관공급업,지주회사,1956-03-03,12월,조원국,http://www.hhic-holdings.com,경기도,D03520


# 상장일자 전처리
* 기존 "YYYY-MM-DD"의 문자형에서 -> "YYYYMMDD"의 숫자형태로 형 변환 진행한 새 Column 생성

In [58]:
# 1) 문자열 정리
s = (
    kind_df2["상장일"]
    .astype(str)
    .str.replace("\n", "", regex=False)
    .str.strip()
)

dt_ymd = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
dt_mdy = pd.to_datetime(s, format="%m-%d-%y", errors="coerce")

dt = dt_ymd.combine_first(dt_mdy)

kind_df2["상장일자"] = dt.dt.strftime("%Y%m%d")

In [59]:
kind_df2[["상장일", "상장일자"]].head(10)

,상장일,상장일자
0,2026-01-14,20260114
1,2026-01-14,20260114
2,2025-12-29,20251229
3,2025-12-29,20251229
4,2025-12-24,20251224
5,2025-12-23,20251223
6,2025-12-22,20251222
7,2025-12-22,20251222
8,2025-12-22,20251222
9,2025-12-22,20251222


In [61]:
kind_df2["상장일자"].isna().sum()

0

In [62]:
today = pd.Timestamp.today().normalize()
listed_date = pd.to_datetime(kind_df2["상장일자"], format="%Y%m%d")

kind_df2["상장경과일"] = (today - listed_date).dt.days
kind_df2["상장연차"] = (kind_df2["상장경과일"] / 365).astype(int)

In [63]:
kind_df2

,회사명,시장구분,종목코드,업종,주요제품,상장일,결산월,대표자명,홈페이지,지역,산업분류코드,상장일자,상장경과일,상장연차
0,엔비알모션,코스닥,0004V0,기타 금속 가공제품 제조업,강구 및 테이퍼롤러,2026-01-14,12월,문두성,NaN,경상남도,C00259,20260114,6,0
1,지에프아이,코스닥,493330,일반 목적용 기계 제조업,"2차전지소화안전제품, 일반소화안전제품",2026-01-14,12월,이상섭,NaN,경기도,C00291,20260114,6,0
2,삼미금속,코스닥,012210,기타 금속 가공제품 제조업,"자동차부품, 중장비부품, 선박부품, 방산부품 등",2025-12-29,12월,이남영,http://www.sammimetal.com,경상남도,C00259,20251229,22,0
3,세미파이브,코스닥,490470,"건축기술, 엔지니어링 및 관련 기술 서비스업",반도체 설계 서비스,2025-12-29,12월,조명현,http://semifive.com,경기도,M00721,20251229,22,0
4,리브스메드,코스닥,491000,의료용 기기 제조업,"다관절 다자유도 복강경 수술기구인 ArtiSential, ArtiSeal, Arti...",2025-12-24,12월,이정주,http://www.livsmed.com,경기도,C00271,20251224,27,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3276,경방,유가,000050,종합 소매업,"섬유류(면사,면혼방사,면직물,면혼방직물,화섬사,화섬직물) 제조,도매,수출입",1956-03-03,12월,"김준, 김담",http://www.kyungbang.co.kr,서울특별시,G00471,19560303,25525,69
3277,유수홀딩스,유가,000700,회사 본부 및 경영 컨설팅 서비스업,지주사업,1956-03-03,12월,송영규,http://www.eusu-holdings.com,서울특별시,M00715,19560303,25525,69
3278,한진중공업홀딩스,유가,003480,연료용 가스 제조 및 배관공급업,지주회사,1956-03-03,12월,조원국,http://www.hhic-holdings.com,경기도,D00352,19560303,25525,69
3279,한진중공업홀딩스,유가,003480,연료용 가스 제조 및 배관공급업,지주회사,1956-03-03,12월,조원국,http://www.hhic-holdings.com,경기도,D03520,19560303,25525,69


In [64]:
kind_df2.to_csv("./data/metadata/company_cord_prototype.csv", encoding='cp949', index=False)